# 05 · Next Best Action

**Data:** `uci_nba_offer_events.parquet` + `uci_customer_features.parquet`  
**(A) Supervised:** multi-class classifier → best `offer_id`  
**(B) RL agent:** tabular Q-learning (state = RFM segment × tier, action = offer type)

Split: 60/20/20 stratified on `converted`.


In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def find_project_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / "scripts" / "build_datasets.py").exists():
            return candidate
    return path


PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data" / "modeling"
MODELS = PROJECT_ROOT / "models"
MODELS.mkdir(exist_ok=True)


def load_parquet(name: str) -> pd.DataFrame:
    path = DATA / name
    if not path.exists():
        raise FileNotFoundError(f"Missing {path} — run: python scripts/uci_pipeline.py")
    return pd.read_parquet(path)


def save_artifact(name: str, obj) -> Path:
    path = MODELS / name
    joblib.dump(obj, path)
    print(f"Saved → {path.relative_to(PROJECT_ROOT)}")
    return path


def audit_and_clean(
    df: pd.DataFrame,
    *,
    subset: list[str] | None = None,
    id_col: str | None = None,
    required_cols: list[str] | None = None,
    label: str = "dataset",
) -> pd.DataFrame:
    """Report and drop duplicate rows + rows with NA in required columns (before split)."""
    out = df.copy()
    n0 = len(out)
    dup_subset = subset if subset is not None else ([id_col] if id_col else None)
    n_dup = out.duplicated(subset=dup_subset, keep="first").sum() if dup_subset else out.duplicated(keep="first").sum()
    if n_dup:
        out = out.drop_duplicates(subset=dup_subset, keep="first")
    req = [c for c in (required_cols or []) if c in out.columns]
    na_rows = out[req].isna().any(axis=1).sum() if req else 0
    na_by_col = out[req].isna().sum()
    if req:
        out = out.dropna(subset=req)
    print(
        f"[{label}] {n0:,} rows -> {len(out):,} | "
        f"dropped {n_dup:,} duplicates, {na_rows:,} rows with NA"
    )
    if na_rows and (na_by_col > 0).any():
        print("  NA counts:", na_by_col[na_by_col > 0].to_dict())
    return out


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
import xgboost as xgb

events = load_parquet("uci_nba_offer_events.parquet")
feat = load_parquet("uci_customer_features.parquet")
catalog = load_parquet("uci_nba_offer_catalog.parquet")

df = events.merge(feat, on="customer_id", how="left", suffixes=("", "_feat"))
df = df[df["shown"] == 1].copy()  # only shown offers

FEATURES = [
    "RFM_score", "engagement_score", "discount_sensitivity",
    "recency_days", "discount_pct", "clicked",
]
cat_features = ["rfm_segment", "tier", "channel", "offer_type"]
df = audit_and_clean(
    df,
    subset=["customer_id", "offer_id", "event_date", "channel"],
    required_cols=["customer_id", "offer_id", "converted", *FEATURES, *cat_features],
    label="nba_offer_events",
)
print(f"Shown offers: {len(df):,} · Conversion rate: {df['converted'].mean():.1%}")


In [ ]:
num = df[FEATURES + ["converted"]].corr()
sns.heatmap(num, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("NBA numeric feature correlation")
plt.tight_layout()
plt.show()


In [ ]:
# (A) Supervised multi-class: predict offer_id for converted=1 subset + weighted
le = LabelEncoder()
df["offer_id_enc"] = le.fit_transform(df["offer_id"])

X_num = df[FEATURES]
X_cat = pd.get_dummies(df[cat_features], drop_first=False)
X_all = pd.concat([X_num, X_cat], axis=1)
y_cls = df["offer_id_enc"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_cls, test_size=0.40, stratify=y_cls, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

candidates = {
    "XGBoost": xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        objective="multi:softprob", num_class=len(le.classes_),
        random_state=RANDOM_STATE, verbosity=0,
    ),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1),
}

sup_results = []
for name, clf in candidates.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    sup_results.append({
        "model": name,
        "test_accuracy": accuracy_score(y_test, pred),
        "test_f1_macro": f1_score(y_test, pred, average="macro"),
        "estimator": clf,
    })

sup_lb = pd.DataFrame([{k: v for k, v in r.items() if k != "estimator"} for r in sup_results]).set_index("model")
display(sup_lb.round(4))
best_sup = max(sup_results, key=lambda r: r["test_f1_macro"])
print(f"Best supervised: {best_sup['model']}")


In [ ]:
# (B) Tabular Q-learning for offer_type selection
actions = sorted(df["offer_type"].unique())
state_keys = (df["rfm_segment"].astype(str) + "|" + df["tier"].astype(str)).unique()
Q = {s: {a: 0.0 for a in actions} for s in state_keys}

alpha, gamma, epsilon = 0.1, 0.9, 0.15
rng = np.random.default_rng(RANDOM_STATE)

def reward(row):
    profit = 1.0 if row["converted"] else 0.0
    cost = row["discount_pct"] * 0.5 + (0.1 if row["channel"] == "email" else 0.05)
    retention = 0.3 if row["rfm_segment"] in ("At Risk", "Hibernating") and row["converted"] else 0.0
    return profit + retention - cost

train_df = df.sample(frac=0.7, random_state=RANDOM_STATE)
for _, row in train_df.iterrows():
    s = f"{row['rfm_segment']}|{row['tier']}"
    a = row["offer_type"]
    r = reward(row)
    if rng.random() < epsilon:
        a = rng.choice(actions)
    Q[s][a] = Q[s][a] + alpha * (r - Q[s][a])

def rl_recommend(state):
    if state not in Q:
        return actions[0]
    return max(Q[state], key=Q[state].get)

test_df = df.drop(train_df.index)
hits = sum(rl_recommend(f"{r['rfm_segment']}|{r['tier']}") == r["offer_type"] for _, r in test_df.iterrows())
print(f"RL policy match rate on holdout: {hits/len(test_df):.1%}")


In [ ]:
save_artifact("05_nba_supervised_best.joblib", {
    "model_name": best_sup["model"],
    "model": best_sup["estimator"],
    "label_encoder": le,
    "feature_columns": list(X_all.columns),
    "metrics": {k: best_sup[k] for k in ("test_accuracy", "test_f1_macro")},
})
save_artifact("05_nba_rl_qtable.joblib", {
    "Q": Q,
    "actions": actions,
    "alpha": alpha,
    "gamma": gamma,
})
